In [ ]:
import os
import pandas as pd
from datasets import Dataset

from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card

client, PROJECT_NAME = get_client()

In [ ]:
# Microbiology events for all cohort patients across their full relevant stay window:
#   - ED-only patients (no hadm_id): ED visit window only (intime → outtime)
#   - Admitted, no ICU: full hospital stay (ed_intime → dischtime)
#   - Admitted, went to ICU: ED intime up to ICU transfer (ed_intime → first_icu_intime)
# result_available_in_er flags the ~1.2% of cultures whose results came back before ED discharge

query_micro = f"""
SELECT
  c.subject_id,
  c.hadm_id,
  c.ed_stay_id,
  m.microevent_id,
  m.micro_specimen_id,
  m.charttime,
  m.spec_itemid,
  m.spec_type_desc,
  m.test_itemid,
  m.test_name,
  m.org_name,
  m.comments,
  m.storetime,
  IF(m.storetime IS NOT NULL
     AND CAST(m.storetime AS DATETIME) < c.ed_outtime, 1, 0) AS result_available_in_er,
  c.disposition,
  c.cohort_label
FROM `{PROJECT_NAME}.rl_project.cohort_base` c
INNER JOIN `physionet-data.mimiciv_3_1_hosp.microbiologyevents` m
  ON c.subject_id = m.subject_id
WHERE m.charttime IS NOT NULL
  AND m.charttime >= c.ed_intime
  AND m.charttime < CASE
    -- Admitted patients who went to ICU: cut off at ICU transfer
    WHEN c.hadm_id IS NOT NULL AND c.first_icu_intime IS NOT NULL
      THEN c.first_icu_intime
    -- Admitted patients who never went to ICU: full hospital stay
    WHEN c.hadm_id IS NOT NULL AND c.first_icu_intime IS NULL
      THEN c.dischtime
    -- ED-only patients: ED window only
    ELSE c.ed_outtime
  END
ORDER BY c.subject_id, c.ed_stay_id, m.charttime
"""

df_micro = query_to_df(client, query_micro)

admitted = df_micro[df_micro['hadm_id'].notna()]
home     = df_micro[df_micro['hadm_id'].isna()]

print(f"Total rows          : {len(df_micro):,}")
print(f"Unique ED stays     : {df_micro['ed_stay_id'].nunique():,}")
print(f"Unique subjects     : {df_micro['subject_id'].nunique():,}")
print()
print(f"Admitted rows       : {len(admitted):,}  |  unique stays: {admitted['ed_stay_id'].nunique():,}")
print(f"Discharged home rows: {len(home):,}  |  unique stays: {home['ed_stay_id'].nunique():,}")
print()
rows_per_stay = df_micro.groupby('ed_stay_id').size()
print(f"Stays with 1 culture row  : {(rows_per_stay == 1).sum():,}")
print(f"Stays with 2+ culture rows: {(rows_per_stay > 1).sum():,}")
print(f"Max culture rows per stay : {rows_per_stay.max():,}")
print()
print(f"Result available in ER: {df_micro['result_available_in_er'].sum():,} ({df_micro['result_available_in_er'].mean()*100:.1f}%)")
df_micro.head(20)

In [ ]:
DESCRIPTION = (
    "Microbiology culture events from hosp.microbiologyevents for cohort patients during their stay window. "
    "Includes culture order time, specimen type, organism name, and antibiotic sensitivity results. "
    "culture_ordered is the real-time state signal; culture_positive is a retrospective label only "
    "(~2% of results available before ED discharge)."
)

ds = Dataset.from_pandas(df_micro, split='microbiology_events')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="microbiology_events", data_dir="microbiology")
push_dataset_card("ADS599-Capstone/raw_data", config_name="microbiology_events", split="microbiology_events", data_dir="microbiology", description=DESCRIPTION)
print("Pushed microbiology_events to HuggingFace Hub.")